## Model Traning 

In [33]:
# # ==============================
# # TRAIN MODEL 
# # ==============================

import pandas as pd
import numpy as np
import joblib

# Added more robust metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# ------------------------------
# Load Processed Data
# ------------------------------
X_train = pd.read_csv('../../data/processed/X_train.csv')
X_test = pd.read_csv('../../data/processed/X_test.csv')
y_train = pd.read_csv('../../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../../data/processed/y_test.csv').values.ravel()

# ------------------------------
# 1. Logistic Regression (Baseline)
# ------------------------------
lr = LogisticRegression(max_iter=1000, class_weight='balanced') # Added class_weight
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# ------------------------------
# 2. Decision Tree (Regulated)
# ------------------------------
# IMPROVEMENT: Added max_depth and min_samples_leaf to prevent overfitting/memorization
dt = DecisionTreeClassifier(
    max_depth=5, 
    min_samples_leaf=10, 
    random_state=42,
    class_weight='balanced'
)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# ------------------------------
# 3. Random Forest (Robust)
# ------------------------------
# IMPROVEMENT: Restricted depth and increased estimators for better generalization
rf = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# ------------------------------
# Enhanced Evaluation Function
# ------------------------------
def evaluate_model(name, y_true, y_pred, model, X):
    print(f"\n===== {name} Evaluation =====")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_true, y_pred, average='weighted'):.4f}") # Better for imbalanced data
    
    # Cross-Validation to ensure stability
    cv_scores = cross_val_score(model, X, y_true, cv=5)
    print(f"Cross-Val Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.2f})")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

evaluate_model("Logistic Regression", y_test, y_pred_lr, lr, X_test)
evaluate_model("Decision Tree", y_test, y_pred_dt, dt, X_test)
evaluate_model("Random Forest", y_test, y_pred_rf, rf, X_test)

# ------------------------------
# Feature Importance & Selection
# ------------------------------
# We use RF as it's usually the most reliable for importance
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Features (Generalizable):")
print(feature_importance.head(10))

# ------------------------------
# Save Model
# ------------------------------
joblib.dump(rf, '../models/employee_performance_model.pkl')
print("\n✅ Robust Model saved successfully!")


===== Logistic Regression Evaluation =====
Accuracy: 0.8457
F1-Score: 0.8454
Cross-Val Mean: 0.8057 (+/- 0.07)

Confusion Matrix:
[[155  16   4]
 [ 16 140  19]
 [ 14  12 149]]

===== Decision Tree Evaluation =====
Accuracy: 0.9219
F1-Score: 0.9202
Cross-Val Mean: 0.8914 (+/- 0.03)

Confusion Matrix:
[[172   2   1]
 [ 18 141  16]
 [  2   2 171]]

===== Random Forest Evaluation =====
Accuracy: 0.9924
F1-Score: 0.9924
Cross-Val Mean: 0.9581 (+/- 0.04)

Confusion Matrix:
[[173   2   0]
 [  2 173   0]
 [  0   0 175]]

Top 10 Features (Generalizable):
                         Feature  Importance
11      EmpLastSalaryHikePercent    0.220990
4     EmpEnvironmentSatisfaction    0.217289
18       YearsSinceLastPromotion    0.082403
17  ExperienceYearsInCurrentRole    0.038886
15            EmpWorkLifeBalance    0.034211
5                  EmpHourlyRate    0.033372
16  ExperienceYearsAtThisCompany    0.032758
0                            Age    0.027211
13    TotalWorkExperienceInYears    0.0257

In [30]:
# Train accuracy
train_pred = rf.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)

# Test accuracy
test_acc = accuracy_score(y_test, y_pred_rf)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 1.0
Test Accuracy: 0.9942857142857143


In [31]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(rf, X_train, y_train, cv=5)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Score:", cv_scores.mean())

Cross Validation Scores: [0.99047619 0.99047619 0.98329356 0.97852029 0.9928401 ]
Mean CV Score: 0.9871212637799751


# Model Summary & Insights

## 1. Model Performance Overview
We evaluated three classification models to predict employee performance. The **Random Forest** emerged as the superior model, demonstrating high stability and generalization across cross-validation folds.

| Model | Test Accuracy | F1-Score (Weighted) | CV Mean (5-Fold) |
| :--- | :--- | :--- | :--- |
| **Logistic Regression** | 84.57% | 0.8454 | 80.57% |
| **Decision Tree** | 92.19% | 0.9202 | 89.14% |
| **Random Forest** | **99.24%** | **0.9924** | **95.81%** |

---

## 2. Key Technical Findings
* **Non-Linearity:** The significant performance jump from Logistic Regression (84%) to Tree-based models (92%+) indicates that the relationships in this dataset are non-linear and involve complex feature interactions.
* **Model Robustness:** The Random Forest's high Cross-Validation score (**95.81%**) suggests that while the accuracy is near-perfect, the model is consistent across different data subsets and not merely overfitting to a single training split.
* **Error Analysis:** According to the Confusion Matrix, the model is exceptionally balanced, with nearly equal precision and recall across all classes.

---

## 3. Business Drivers (Feature Importance)
The model identified the following as the top drivers for employee outcomes:

1.  **EmpLastSalaryHikePercent (~22.1%):** The most critical predictor, suggesting financial growth is a primary driver.
2.  **EmpEnvironmentSatisfaction (~21.7%):** Nearly as important as salary, emphasizing the role of workplace culture.
3.  **YearsSinceLastPromotion (~8.2%):** Indicates that career stagnation is a secondary but significant risk factor.

---

## 4. Recommendations & Next Steps
* **Deployment:** The Random Forest model is recommended for production use due to its high F1-Score and reliability.
* **Strategy:** Management should focus on **Salary Hike transparency** and **Environment Satisfaction surveys**, as these two factors alone control over 40% of the model's decision-making logic.
* **Future Work:** Investigate the specific threshold of *YearsSinceLastPromotion* to create a "Red Flag" alert system for HR to intervene before performance drops.
